<a href="https://colab.research.google.com/github/ebritolbv-cmd/BrazilQuantumCamp/blob/main/dados_para_aula.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## F1 — Risco de Inadimplência (Classificação Binária)

### 🎯 Objetivo
Prever se um cliente terá risco de inadimplência (0 = não, 1 = sim).

### 📊 O que a função `prep_F1()` faz?

1. Gera 5 variáveis financeiras simuladas:
   - `dti` → dívida / renda
   - `util` → utilização do limite
   - `late` → histórico de atrasos
   - `income` → renda normalizada
   - `savings` → reserva financeira

2. Cria um score de risco usando uma combinação linear dessas variáveis + ruído.

3. Converte o score em classe binária (aprox. 30% dos casos são risco alto).

4. Divide os dados em:
   - treino (80%)
   - teste (20%)

5. Normaliza as features (usando apenas dados de treino).

6. Retorna tensores prontos para usar no PyTorch/QNN.

---

### 🧠 Como usar

```python
Xtr, ytr, Xte, yte = prep_F1()


In [ ]:
import numpy as np
import torch

def prep_F1(n=2000, test_ratio=0.2, seed=0):
    rng = np.random.default_rng(seed)

    # 5 features em [0,1]
    dti = rng.uniform(0, 1, n)             # debt-to-income
    util = rng.uniform(0, 1, n)            # uso do limite
    late = rng.uniform(0, 1, n)            # atrasos (proxy)
    income = rng.uniform(0, 1, n)          # renda (norm)
    savings = rng.uniform(0, 1, n)         # reserva (norm)

    X = np.stack([dti, util, late, income, savings], axis=1).astype(np.float32)

    # Regra simples: risco aumenta com dti/util/late e diminui com savings
    score = 1.6*dti + 1.2*util + 1.0*late - 1.0*savings + 0.2*rng.normal(size=n)
    y = (score > np.quantile(score, 0.7)).astype(np.float32)  # ~30% positivos

    # split
    idx = np.arange(n); rng.shuffle(idx)
    cut = int(n*(1-test_ratio))
    tr, te = idx[:cut], idx[cut:]
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

    # padroniza (fit no treino)
    mu, sig = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-8
    Xtr = (Xtr - mu)/sig
    Xte = (Xte - mu)/sig

    return (torch.tensor(Xtr), torch.tensor(ytr),
            torch.tensor(Xte), torch.tensor(yte))


---

# 📘 F3 — Regressão de Gasto Mensal

```markdown
## F3 — Previsão de Gasto do Próximo Mês (Regressão)

### 🎯 Objetivo
Prever o valor contínuo do gasto no próximo mês.

### 📊 O que a função `prep_F3()` faz?

1. Gera variáveis financeiras:
   - `income` → renda
   - `spend_now` → gasto atual
   - `avg3` → média recente de gastos
   - `credit_ratio` → uso do cartão

2. Cria o gasto do próximo mês como uma combinação dessas variáveis + ruído.

3. Divide em treino e teste.

4. Normaliza:
   - Features X
   - Alvo y (importante para estabilidade na regressão)

5. Retorna tensores prontos.

---

### 🧠 Como usar

```python
Xtr, ytr, Xte, yte = prep_F3()


In [ ]:
import numpy as np
import torch

def prep_F3(n=2000, test_ratio=0.2, seed=0):
    rng = np.random.default_rng(seed)

    # features em [0,1]
    income = rng.uniform(0, 1, n)
    spend_now = rng.uniform(0, 1, n)
    avg3 = 0.6*spend_now + 0.4*rng.uniform(0, 1, n)          # média recente correlacionada
    credit_ratio = rng.uniform(0, 1, n)

    X = np.stack([income, spend_now, avg3, credit_ratio], axis=1).astype(np.float32)

    # alvo contínuo (gasto próximo mês) com ruído
    y = (0.55*spend_now + 0.35*avg3 + 0.20*credit_ratio + 0.10*rng.normal(size=n)).astype(np.float32)

    # split
    idx = np.arange(n); rng.shuffle(idx)
    cut = int(n*(1-test_ratio))
    tr, te = idx[:cut], idx[cut:]
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

    # padroniza X (fit treino)
    mu, sig = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-8
    Xtr = (Xtr - mu)/sig
    Xte = (Xte - mu)/sig

    # normaliza y (opcional, mas ajuda treino)
    ymu, ysig = ytr.mean(), ytr.std() + 1e-8
    ytr = (ytr - ymu)/ysig
    yte = (yte - ymu)/ysig

    return (torch.tensor(Xtr), torch.tensor(ytr),
            torch.tensor(Xte), torch.tensor(yte))



---

# 📘 L1 — Classificação de Atraso

```markdown
## L1 — Previsão de Atraso na Entrega (Classificação Binária)

### 🎯 Objetivo
Prever se uma entrega vai atrasar.

### 📊 O que a função `prep_L1()` faz?

1. Gera variáveis logísticas:
   - `distance` → distância da rota
   - `congestion` → nível de trânsito
   - `weather` → risco climático
   - `hub_load` → carga do centro logístico

2. Combina essas variáveis para gerar probabilidade de atraso.

3. Define atraso como classe 1 (~40% dos casos).

4. Divide em treino/teste.

5. Normaliza as features.

6. Retorna tensores.

---

### 🧠 Como usar

```python
Xtr, ytr, Xte, yte = prep_L1()


In [ ]:
import numpy as np
import torch

def prep_L1(n=2000, test_ratio=0.2, seed=0):
    rng = np.random.default_rng(seed)

    # features em [0,1]
    distance = rng.uniform(0, 1, n)
    congestion = rng.uniform(0, 1, n)
    weather = rng.uniform(0, 1, n)
    hub_load = rng.uniform(0, 1, n)

    X = np.stack([distance, congestion, weather, hub_load], axis=1).astype(np.float32)

    # atraso aumenta com tudo
    score = 1.2*distance + 1.6*congestion + 1.3*weather + 0.8*hub_load + 0.2*rng.normal(size=n)
    y = (score > np.quantile(score, 0.6)).astype(np.float32)  # ~40% atrasos

    # split
    idx = np.arange(n); rng.shuffle(idx)
    cut = int(n*(1-test_ratio))
    tr, te = idx[:cut], idx[cut:]
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

    # padroniza
    mu, sig = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-8
    Xtr = (Xtr - mu)/sig
    Xte = (Xte - mu)/sig

    return (torch.tensor(Xtr), torch.tensor(ytr),
            torch.tensor(Xte), torch.tensor(yte))



---

# 📘 L2 — Regressão de Tempo de Entrega (ETA)

```markdown
## L2 — Previsão de Tempo de Entrega (Regressão)

### 🎯 Objetivo
Prever o tempo estimado de entrega (ETA).

### 📊 O que a função `prep_L2()` faz?

1. Gera variáveis logísticas:
   - `distance`
   - `stops`
   - `congestion`
   - `priority`

2. Calcula ETA como combinação dessas variáveis + ruído.

3. Divide em treino/teste.

4. Normaliza:
   - Features X
   - Alvo y (para facilitar convergência)

5. Retorna tensores prontos.

---

### 🧠 Como usar

```python
Xtr, ytr, Xte, yte = prep_L2()


In [ ]:
import numpy as np
import torch

def prep_L2(n=2000, test_ratio=0.2, seed=0):
    rng = np.random.default_rng(seed)

    # features em [0,1]
    distance = rng.uniform(0, 1, n)
    stops = rng.uniform(0, 1, n)
    congestion = rng.uniform(0, 1, n)
    priority = rng.uniform(0, 1, n)

    X = np.stack([distance, stops, congestion, priority], axis=1).astype(np.float32)

    # ETA aumenta com distância/paradas/trânsito e diminui com prioridade
    y = (0.9*distance + 0.4*stops + 1.1*congestion - 0.3*priority + 0.1*rng.normal(size=n)).astype(np.float32)

    # split
    idx = np.arange(n); rng.shuffle(idx)
    cut = int(n*(1-test_ratio))
    tr, te = idx[:cut], idx[cut:]
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

    # padroniza X
    mu, sig = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-8
    Xtr = (Xtr - mu)/sig
    Xte = (Xte - mu)/sig

    # normaliza y (ajuda treino)
    ymu, ysig = ytr.mean(), ytr.std() + 1e-8
    ytr = (ytr - ymu)/ysig
    yte = (yte - ymu)/ysig

    return (torch.tensor(Xtr), torch.tensor(ytr),
            torch.tensor(Xte), torch.tensor(yte))
